In [0]:
import re
from pyspark.sql import functions as F

CATALOG ="global_electronics_retailer_dataset"
RAW = f"/Volumes/{CATALOG}/bronze/raw_files"

Checking raw files in bronze volume

In [0]:
display(dbutils.fs.ls(RAW))

Load as Dataframe

In [0]:
df= spark.read\
.option("header", True)\
.option("inferSchema", False)\
.csv(f"{RAW}/Sales.csv")

df.printSchema()

display(df.limit(5))


In [0]:
def to_snakecase(col):
    col = re.sub(r"(?<=[a-z0-9])(?=[A-Z])", "_", col.strip())  # CustomerKey -> Customer_Key
    col = re.sub(r"[^0-9a-zA-Z]+", "_", col)                   # spaces/symbols -> _
    return col.strip("_").lower()

df = df.toDF(*[to_snakecase(col)for col in df.columns])

display(df.limit(3))

add metadata columns , timestamp and filepath

In [0]:
df = df.withColumn("_sourcefile", F.col("_metadata.file_path"))\
.withColumn("_ingest_time", F.current_timestamp())

display(df.limit(3))

write to a  delta table

In [0]:
df.write\
.mode("overwrite")\
.option("overwriteSchema", True)\
.saveAsTable(f"{CATALOG}.bronze.sales")

Add rest of the files as well

In [0]:
skip = {"Data_Dictionary.csv"}
for f in dbutils.fs.ls(RAW):
    if not f.name.endswith(".csv") or f.name in skip :
        continue
    table = to_snakecase(f.name.removesuffix(".csv"))
    df=spark.read\
        .option("header", True)\
        .option("inferSchema", False)\
        .option("encoding", "ISO-8859-1")\
        .csv(f"{RAW}/{f.name}")
    df = df.toDF(*[to_snakecase(col)for col in df.columns])
    df = df.withColumn("_sourcefile",F.col("_metadata.file_path"))\
        .withColumn("_ingest_time",F.current_timestamp())
    df.write\
        .mode("overwrite")\
        .option("overwriteSchema", True)\
        .saveAsTable(f"{CATALOG}.bronze.{table}")


Bronze table validation

In [0]:
for t in ["sales", "customers", "products", "stores", "exchange_rates"]:
    n= spark.table(f"{CATALOG}.bronze.{t}").count()
    print(f"{t} has {n} records")